## Setup start 

In [ ]:
source("~/workspace/pipelines/snt_dhis2_formatting/utils/snt_dhis2_formatting.r")
snt_paths <- init_snt_workspace(
    snt_pipeline_name="snt_dhis2_formatting",
    packages=c("arrow", "dplyr", "tidyr", "stringr", "stringi", "jsonlite", "httr", "glue"))

# Load config
config_json <- load_snt_config(file.path(snt_paths$CONFIG_PATH, "SNT_config.json"))

# Save config variables
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
ADMIN_1 <- toupper(config_json$SNT_CONFIG$DHIS2_ADMINISTRATION_1)
ADMIN_2 <- toupper(config_json$SNT_CONFIG$DHIS2_ADMINISTRATION_2)
extracts_dataset_id <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_EXTRACTS

### Load DHIS2 organisation units data

-Load DHIS2 organisation units from latest dataset version 


In [ ]:
dhis2_pyramid_data <- load_dataset_file(extracts_dataset_id, paste0(COUNTRY_CODE, "_dhis2_raw_pyramid.parquet"), verbose=FALSE)
log_msg(glue("DHIS2 organisation units data loaded from dataset : '{extracts_dataset_id}' dataframe dimensions: {paste(dim(dhis2_pyramid_data), collapse=', ')}"))
head(dhis2_pyramid_data, 3)

### Load DHIS2 population data

-Load DHIS2 population from latest dataset version 


In [ ]:
dhis2_data <- load_dataset_file(extracts_dataset_id, paste0(COUNTRY_CODE, "_dhis2_raw_population.parquet"), verbose=FALSE)
log_msg(glue("DHIS2 population data loaded from dataset : '{extracts_dataset_id}' dataframe dimensions: {paste(dim(dhis2_pyramid_data), collapse=', ')}"))

head(dhis2_data,3)

## SNT total population aggregation  

-Build Population indicators based on configuration definitions

In [ ]:
population_table <- build_population_indicators(dhis2_data, dhis2_pyramid_data, config_json)
print(dim(population_table))
head(population_table, 3)

## Format SNT population data  

-Apply standard SNT formatting for the final table

In [ ]:
admin_cols <- get_admin_config(config_json)
population_table_formatted <- standardize_population_columns(population_table, admin_cols)

log_msg("Formatting population SNT data finished.")
print(dim(population_table_formatted))
head(population_table_formatted,3)

### Create population template

In [ ]:
# Disaggregation columns
disaggregation_cols <- c("POP_UNDER_5", "POP_PREGNANT_WOMEN", "POP_0_1_Y", "POP_1_2_Y", "POP_5_10_Y", "POP_5_36_M", "POP_50_PLUS")

# Create pop template (util for pop transformation pipeline)
pop_template <- population_table_formatted %>% 
    select(ADM1_NAME, ADM1_ID, ADM2_NAME, ADM2_ID) %>%
    distinct() %>%
    arrange(ADM1_NAME, ADM2_NAME) %>%    
    mutate(!!!setNames(rep(list(NA_real_), length(disaggregation_cols)), disaggregation_cols)) # Numeric

head(pop_template, 3)

### Output data

In [ ]:
FORMATTED_DATA_PATH <- file.path(snt_paths$DATA_PATH, "dhis2", "extracts_formatted")

# write files
write_parquet(population_table_formatted, file.path(FORMATTED_DATA_PATH, paste0(COUNTRY_CODE, "_population.parquet")))
write.csv(population_table_formatted, file.path(FORMATTED_DATA_PATH, paste0(COUNTRY_CODE, "_population.csv")), row.names = FALSE)

# log
log_msg(glue("Population data saved under: {file.path(FORMATTED_DATA_PATH, paste0(COUNTRY_CODE, '_population.csv'))}"))

In [ ]:
# save template
write.csv(pop_template, file.path(snt_paths$UPLOADS_PATH, paste0(COUNTRY_CODE, "_population_template.csv")), row.names = FALSE)
log_msg(glue("Population template created : {file.path(snt_paths$UPLOADS_PATH, paste0(COUNTRY_CODE, '_population_template.csv'))}"))

### Data Summary 

In [ ]:
# Data summary
print(summary(population_table_formatted))